
Привет, меня зовут Люман Аблаев. Сегодня я проверю твой проект.
<br> Дальнейшее общение будет происходить на "ты" если это не вызывает никаких проблем.
<br> Желательно реагировать на красные комментарии ('исправил', 'не понятно как исправить ошибку', ...)
<br> Пожалуйста, не удаляй комментарии ревьюера, так как они повышают качество повторного ревью.

Комментарии будут в <font color='green'>зеленой</font>, <font color='blue'>синей</font> или <font color='red'>красной</font> рамках:


<div class="alert alert-block alert-success">
<b>Успех:</b> Если все сделано отлично
</div>

<div class="alert alert-block alert-info">
<b>Совет: </b> Если можно немного улучшить
</div>

<div class="alert alert-block alert-danger">
<b>Ошибка:</b> Если требуются исправления. Работа не может быть принята с красными комментариями.
</div>

-------------------

Будет очень хорошо, если ты будешь помечать свои действия следующим образом:
<div class="alert alert-block alert-warning">
<b>Комментарий студента:</b> ...
</div>

<div class="alert alert-block alert-warning">
<b>Изменения:</b> Были внесены следующие изменения ...
</div>







<font color='orange' style='font-size:24px; font-weight:bold'>Общее впечатление</font>
* Спасибо за очень качественную работу - видно, что вложено много усилий.
- Я оставил некоторые советы, надеюсь они будут полезными и интересными
- Есть некоторые недочеты, которые нужно поправить, но у тебя это не должно занять много времени)
- Жду обновленную работу






# Анализ лояльности пользователей Яндекс Афиши



 <div class="alert alert-block alert-info">
<b>Совет:</b> Пожалуйста, формируй полностью вводную часть - это важная составляющие любой работы.  Нужны цели и задачи, описание данных, содержание проекта.
</div>




## Этапы выполнения проекта

### 1. Загрузка данных и их предобработка

---

**Задача 1.1:** Напишите SQL-запрос, выгружающий в датафрейм pandas необходимые данные. Используйте следующие параметры для подключения к базе данных `data-analyst-afisha`:

- **Хост** — `rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net`
- **База данных** — `data-analyst-afisha`
- **Порт** — `6432`
- **Аутентификация** — `Database Native`
- **Пользователь** — `praktikum_student`
- **Пароль** — `Sdf4$2;d-d30pp`

Для выгрузки используйте запрос из предыдущего урока и библиотеку SQLAlchemy.

Выгрузка из базы данных SQL должна позволить собрать следующие данные:

- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ (`mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk`);
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk`);
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

---


In [1]:
# Используйте ячейки типа Code для вашего кода,
# а ячейки типа Markdown для комментариев и выводов

In [2]:
# При необходимости добавляйте новые ячейки для кода или текста

In [3]:
!pip install sqlalchemy psycopg2-binary

In [4]:
import pandas as pd
from sqlalchemy import create_engine
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
!pip install python-dotenv

In [6]:
from dotenv import load_dotenv
import os

# Загрузка .env
load_dotenv()

# Конфиг
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PWD'),
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT')),
    'db': os.getenv('DB_NAME')
}

TypeError: int() argument must be a string, a bytes-like object or a number, not 'NoneType'

             
<div class="alert alert-block alert-success">
    
<b>Успех:</b> Импорты и настройки на месте
</div>



In [ ]:
query = '''
SELECT 
    p.user_id,
    p.device_type_canonical,
    p.order_id,
    p.created_dt_msk AS order_dt,
    p.created_ts_msk AS order_ts,
    p.currency_code,
    p.revenue,
    p.tickets_count,
    (p.created_dt_msk::date - LAG(p.created_dt_msk::date) OVER (PARTITION BY p.user_id ORDER BY p.created_dt_msk)) AS days_since_prev,
    p.event_id,
    p.service_name,
    e.event_type_main,
    r.region_name,
    c.city_name
FROM afisha.purchases p
LEFT JOIN afisha.events e ON p.event_id = e.event_id
LEFT JOIN afisha.city c ON e.city_id = c.city_id
LEFT JOIN afisha.regions r ON c.region_id = r.region_id
WHERE p.device_type_canonical IN ('mobile', 'desktop')
  AND e.event_type_main != 'фильм'
ORDER BY p.user_id ASC
'''

In [ ]:
df = pd.read_sql_query(query, con=engine)

In [ ]:
print(f"Данные успешно выгружены. Размер таблицы: {df.shape}")
print(f"\nПервые 5 строк:")
df.head()


<div class="alert alert-block alert-success">
<b>Успех:</b>  Выгрузка данных проведена корректно! Была выполнена необходимая фильтрация данных, выгружены только необходимые для анализа данные. Отлично, что сразу подсчитываешь время между заказами для каждого пользователя.
</div>
    
<div class="alert alert-block alert-info">
<b>Совет:</b> Можно немного улучшить запрос, добавив также и "ORDER BY order_dt". Так внутри каждого пользователя заказы будут идти строго по дате.
</div>

---

**Задача 1.2:** Изучите общую информацию о выгруженных данных. Оцените корректность выгрузки и объём полученных данных.

Предположите, какие шаги необходимо сделать на стадии предобработки данных — например, скорректировать типы данных.

Зафиксируйте основную информацию о данных в кратком промежуточном выводе.

---

In [ ]:
print(f"\n1. РАЗМЕР ДАННЫХ:")
print(f"   - Количество строк: {df.shape[0]:,}")
print(f"   - Количество столбцов: {df.shape[1]}")

In [ ]:
print(f"\n2. ТИПЫ ДАННЫХ И ПРОПУСКИ:")
print(f"\n   {'Столбец':<25} {'Тип':<15} {'Пропуски':<12} {'% пропусков':<12}")
print(f"   {'-'*25} {'-'*15} {'-'*12} {'-'*12}")
for col in df.columns:
    missing = df[col].isnull().sum()
    missing_pct = (missing / len(df)) * 100
    print(f"   {col:<25} {str(df[col].dtype):<15} {missing:<12} {missing_pct:.2f}%")

In [ ]:
print(f"\n3. СТАТИСТИКА ЧИСЛОВЫХ ПОЛЕЙ:")
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    print(f"\n   {col}:")
    print(f"   - Среднее: {df[col].mean():.2f}")
    print(f"   - Медиана: {df[col].median():.2f}")
    print(f"   - Минимум: {df[col].min():.2f}")
    print(f"   - Максимум: {df[col].max():.2f}")
    print(f"   - 99-й перцентиль: {df[col].quantile(0.99):.2f}")

In [ ]:
print(f"\n4. КАТЕГОРИАЛЬНЫЕ ПОЛЯ:")
categorical_cols = ['device_type_canonical', 'currency_code', 'service_name', 'event_type_main', 'region_name', 'city_name']
for col in categorical_cols:
    if col in df.columns:
        unique_count = df[col].nunique()
        print(f"\n   {col}:")
        print(f"   - Уникальных значений: {unique_count}")
        print(f"   - Примеры: {df[col].dropna().head(3).tolist()}")

In [ ]:
print(f"\n5. ПРОВЕРКА УНИКАЛЬНОСТИ:")
print(f"   - Уникальных пользователей: {df['user_id'].nunique():,}")
print(f"   - Уникальных заказов: {df['order_id'].nunique():,}")
print(f"   - Уникальных мероприятий: {df['event_id'].nunique():,}")

In [ ]:
print(f"\n6. ВРЕМЕННОЙ ДИАПАЗОН:")
print(f"   - Первая покупка: {df['order_dt'].min()}")
print(f"   - Последняя покупка: {df['order_dt'].max()}")
print(f"   - Период: {(df['order_dt'].max() - df['order_dt'].min()).days} дней")

In [ ]:
print(f"\n7. ПРОВЕРКА НА ДУБЛИКАТЫ:")
duplicates = df.duplicated().sum()
print(f"   - Полных дубликатов строк: {duplicates}")

In [ ]:
print(f"\n8. РАСПРЕДЕЛЕНИЕ ПО ТИПАМ УСТРОЙСТВ:")
device_dist = df['device_type_canonical'].value_counts()
for device, count in device_dist.items():
    print(f"   - {device}: {count:,} ({count/len(df)*100:.1f}%)")


In [ ]:
print(f"\n9. РАСПРЕДЕЛЕНИЕ ПО ВАЛЮТАМ:")
currency_dist = df['currency_code'].value_counts()
for currency, count in currency_dist.items():
    print(f"   - {currency}: {count:,} ({count/len(df)*100:.1f}%)")

**ПРОМЕЖУТОЧНЫЙ ВЫВОД ПО ЗАДАЧЕ 1.2:**

1. КОРРЕКТНОСТЬ ВЫГРУЗКИ:

- Данные успешно выгружены, все необходимые поля присутствуют

- Отсутствуют критические пропуски в ключевых полях

- Нет дубликатов строк

2. ОБЪЁМ ДАННЫХ:

- Количество записей о покупках: 290 611

- Количество уникальных пользователей: 21 933

- Временной период: с 2024-06-01 по 2024-10-31 (152 дня)

3. НАБЛЮДАЕМЫЕ ОСОБЕННОСТИ:

- Пропуски присутствуют в поле days_since_prev (21 933 пропуска, 7.55% - закономерно для первых покупок)

- Пропуски в полях region_name и city_name отсутствуют (0.00%)

- Основная валюта - RUB (98.3%), также присутствуют заказы в KZT (1.7%)

4. НЕОБХОДИМЫЕ ШАГИ ПРЕДОБРАБОТКИ:

- Привести выручку к единой валюте (RUB) с учётом курса KZT

- Преобразовать типы данных: order_dt и order_ts в datetime (уже преобразованы)

- Оптимизировать числовые типы для экономии памяти

- Проверить и обработать выбросы в revenue и tickets_count (есть отрицательные значения revenue: -90.76; max revenue: 81 174.54 при 99-м перцентиле 4 003.13)

- Привести категориальные значения к единому формату (при необходимости)

- Заполнить или удалить пропуски в географических полях (по ситуации) - пропуски отсутствуют

<div class="alert alert-block alert-success">
<b>Успех:</b>    Первичный анализ данных выполнен, намечены шаги по их обработке. В целом, данные у нас достаточно неплохого качества, можно рассмотреть вариант понижения размерности для отдельных столбцов. В реальной практике это иногда бывает очень полезным) 
</div>


---

###  2. Предобработка данных

Выполните все стандартные действия по предобработке данных:

---

**Задача 2.1:** Данные о выручке сервиса представлены в российских рублях и казахстанских тенге. Приведите выручку к единой валюте — российскому рублю.

Для этого используйте датасет с информацией о курсе казахстанского тенге по отношению к российскому рублю за 2024 год — `final_tickets_tenge_df.csv`. Его можно загрузить по пути `https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')`

Значения в рублях представлено для 100 тенге.

Результаты преобразования сохраните в новый столбец `revenue_rub`.

---


1. Загружаем данные с курсами валют:

In [ ]:
tenge_kurs = pd.read_csv('https://code.s3.yandex.net/datasets/final_tickets_tenge_df.csv')

Посмотрим на колонки, чтобы понять, как соединять:

In [ ]:
tenge_kurs['data'] = pd.to_datetime(tenge_kurs['data'])

2. Соединяем наш основной df с курсами валют:

In [ ]:
df = df.merge(tenge_kurs[['data', 'curs']], 
              left_on='order_dt', 
              right_on='data', 
              how='left')

3. Пишем функцию для расчета или считаем через условие:

In [ ]:
def convert_to_rub(row):
    if row['currency_code'] == 'KZT':
        return row['revenue'] * (row['curs'] / 100)
    return row['revenue']

<div class="alert alert-block alert-info">
<b>Совет:</b>
    
Существует довольно удобный метод where. Мы можем применить его к столбцу и указать условие, которое будем проверять, а также альтернативный вариант значения. Если условие выполняется, то берется значение из столбца, если нет - альтернативное значение. Тогда расчет выручки в рублях будет выглядеть следующим образом:
    
```
df['revenue_rub'] = df['revenue'].where(df['currency_code'] == 'rub', df['revenue'] * df['curs'] / 100)
```
</div>



Создаем новый столбец:

In [ ]:
df['revenue_rub'] = df.apply(convert_to_rub, axis=1)

Удаляем лишнюю колонку с датой от мерджа, чтобы не мешалась:

In [ ]:
df = df.drop(columns=['data', 'curs'])

Проверяем результат:

In [ ]:
print("Первые 5 строк с новым столбцом revenue_rub:")
display(df[['currency_code', 'revenue', 'revenue_rub']].head())

Проверяем пропуски:

In [ ]:
print(f"Пропусков в revenue_rub: {df['revenue_rub'].isna().sum()}")

---

**Задача 2.2:**

- Проверьте данные на пропущенные значения. Если выгрузка из SQL была успешной, то пропуски должны быть только в столбце `days_since_prev`.
- Преобразуйте типы данных в некоторых столбцах, если это необходимо. Обратите внимание на данные с датой и временем, а также на числовые данные, размерность которых можно сократить.
- Изучите значения в ключевых столбцах. Обработайте ошибки, если обнаружите их.
    - Проверьте, какие категории указаны в столбцах с номинальными данными. Есть ли среди категорий такие, что обозначают пропуски в данных или отсутствие информации? Проведите нормализацию данных, если это необходимо.
    - Проверьте распределение численных данных и наличие в них выбросов. Для этого используйте статистические показатели, гистограммы распределения значений или диаграммы размаха.
        
        Важные показатели в рамках поставленной задачи — это выручка с заказа (`revenue_rub`) и количество билетов в заказе (`tickets_count`), поэтому в первую очередь проверьте данные в этих столбцах.
        
        Если обнаружите выбросы в поле `revenue_rub`, то отфильтруйте значения по 99 перцентилю.

После предобработки проверьте, были ли отфильтрованы данные. Если были, то оцените, в каком объёме. Сформулируйте промежуточный вывод, зафиксировав основные действия и описания новых столбцов.

---

1. Проверка пропусков:

In [ ]:
print(df.isnull().sum())
print("-" * 30)

2. Преобразование типов данных:

In [ ]:
df['order_dt'] = pd.to_datetime(df['order_dt'])
df['order_ts'] = pd.to_datetime(df['order_ts'])

df['tickets_count'] = df['tickets_count'].astype('int32')
df['revenue_rub'] = df['revenue_rub'].astype('float32')

print("Типы данных после преобразования:")
print(df.dtypes.head())
print("-" * 30)

3. Изучение категорий:

In [ ]:
categorical_fields = ['device_type_canonical', 'service_name', 'event_type_main']
for col in categorical_fields:
    print(f"Уникальные значения в {col}: {df[col].unique()}")
print("-" * 30)

4. Обработка ошибок и выбросов:

In [ ]:
rows_before = len(df)
df = df[df['revenue_rub'] >= 0]
revenue_99 = df['revenue_rub'].quantile(0.99)
print(f"99-й перцентиль выручки: {revenue_99:.2f}")
df = df[df['revenue_rub'] <= revenue_99]
print(f"Макс. билетов в заказе: {df['tickets_count'].max()}")

5. Оценка объема удаленных данных:

In [ ]:
rows_after = len(df)
removed_rows = rows_before - rows_after
removed_pct = (removed_rows / rows_before) * 100

print(f"Удалено строк: {removed_rows}")
print(f"Процент удаленных данных: {removed_pct:.2f}%")

6. Визуализация (Гистограмма для проверки):

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['revenue_rub'], bins=50, kde=True)
plt.title('Распределение выручки в рублях (после фильтрации)')
plt.xlabel('Выручка (руб.)')
plt.ylabel('Количество заказов')
plt.show()

**Промежуточный вывод по задаче 2.2:**
- Пропуски: Подтверждено, что пропуски остались только в столбце days_since_prev. Это нормально, так как у пользователей с одной покупкой нет «предыдущей» даты.
- Типы данных: Столбцы с датами приведены к формату datetime. Числовые типы (tickets_count, revenue_rub) оптимизированы для снижения нагрузки на память.
- Очистка от ошибок:
Удалены записи с отрицательной выручкой (технические сбои).
Проведена фильтрация по 99-му перцентилю выручки. Это позволило избавиться от аномально дорогих заказов, которые могли бы исказить средние значения.
- Результат: Суммарно было удалено 1.13% данных, что является допустимым и не влияет на общую репрезентативность выборки.
- Новые столбцы:
revenue_rub — очищенная и переведенная в рубли сумма заказа, готовая для финансового анализа.


<div class="alert alert-block alert-info">
   
<b>Совет:</b>  В подобных случаях хорошо бы оставлять небольшой комментарий, по тому, какие гипотезы можно выдвинуть, с чем связаны эти аномалии. Условно, что мы видим не ошибки в данных (сборе данных), а длинный хвост, то есть какие-то массовые покупки и тп. Они нам не нужны в контексте задачи.
</div>

---

### 3. Создание профиля пользователя

В будущем отдел маркетинга планирует создать модель для прогнозирования возврата пользователей. Поэтому сейчас они просят вас построить агрегированные признаки, описывающие поведение и профиль каждого пользователя.

---

**Задача 3.1.** Постройте профиль пользователя — для каждого пользователя найдите:

- дату первого и последнего заказа;
- устройство, с которого был сделан первый заказ;
- регион, в котором был сделан первый заказ;
- билетного партнёра, к которому обращались при первом заказе;
- жанр первого посещённого мероприятия (используйте поле `event_type_main`);
- общее количество заказов;
- средняя выручка с одного заказа в рублях;
- среднее количество билетов в заказе;
- среднее время между заказами.

После этого добавьте два бинарных признака:

- `is_two` — совершил ли пользователь 2 и более заказа;
- `is_five` — совершил ли пользователь 5 и более заказов.

**Рекомендация:** перед тем как строить профиль, отсортируйте данные по времени совершения заказа.

---


1. Сортировка:

In [ ]:
df = df.sort_values(by=['user_id', 'order_ts'])

2. Создание профиля через группировку:

In [ ]:
users_profiles = df.groupby('user_id').agg(
    first_order_dt=('order_dt', 'min'),
    last_order_dt=('order_dt', 'max'),
    total_orders=('order_id', 'count'),
    avg_revenue_rub=('revenue_rub', 'mean'),
    avg_tickets_count=('tickets_count', 'mean'),
    avg_days_since_prev=('days_since_prev', 'mean')
).reset_index()

3. Вычленение хар-тик первого заказа + добавляем данные из первого заказа:

In [ ]:
first_orders = df.drop_duplicates('user_id').set_index('user_id')
users_profiles['first_device'] = users_profiles['user_id'].map(first_orders['device_type_canonical'])
users_profiles['first_region'] = users_profiles['user_id'].map(first_orders['region_name'])
users_profiles['first_service'] = users_profiles['user_id'].map(first_orders['service_name'])
users_profiles['first_event_type'] = users_profiles['user_id'].map(first_orders['event_type_main'])

4. Добавляем флаги:

In [ ]:
users_profiles['is_two'] = (users_profiles['total_orders'] >= 2).astype(int)
users_profiles['is_five'] = (users_profiles['total_orders'] >= 5).astype(int)


Проверяем:

In [ ]:

print(f"Всего пользователей (до фильтрации): {len(users_profiles)}")
print(f"Совершили 2+ заказа: {users_profiles['is_two'].sum()}")
print(f"Совершили 5+ заказов: {users_profiles['is_five'].sum()}")


<div class="alert alert-block alert-success">
<b>Успех:</b> Профиль пользователя собран, добавлены новые признаки.

</div>

<div class="alert alert-block alert-info">
<b>Совет:</b>   Могу показать и такой вариант формирования профиля здесь:
    
    profiles = (df
            # В начале сортируем данные по дате совершения заказа, что найти первые признаки:
            .sort_values(by='order_ts')
            # Затем группируем по номеру пользователя и агрегируем данные:
            .groupby('user_id')
            .agg(
                # Находим первую и последнюю даты заказа:
                first_order_dt=('order_dt','min'),
                last_order_dt=('order_dt','max'),
                # Находим устройства, регион и название билетного партнера первого заказа:
                first_device=('device_type_canonical','first'),
                first_region_name=('region_name','first'),
                first_service_name=('service_name','first'),
                # Жанр первого посещённого мероприятия (event_type_main):
                first_event_type=('event_type_main','first'),
                # Подсчитваем количество заказов:
                total_orders=('order_id','nunique'),
                # Считаем статистику по заказам: средняя стоимость заказа, среднее количество билетов:
                avg_revenue_rub=('revenue_rub','mean'),
                avg_tickets_count=('tickets_count','mean'),
                # Считаем среднее количество дней между покупками:
                avg_days_since_prev=('days_since_prev','mean')
            )
            # Создаем два признака: совершил ли пользователь 2 / 5 и более заказов:
            .assign(
                is_two = lambda x: x['total_orders'] >= 2,
                is_five = lambda x: x['total_orders'] >= 5
            )
            # Можно альтернативным образом подсчитать среднее количество дней между заказами (если не будет в SQL):
            .assign(
                avg_days = lambda x: (x['last_order_dt'] - x['first_order_dt']).dt.days / (x['total_orders'] - 1)
            )
            .reset_index()
)

Почитать про assign более подробно можно [здесь](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)
</div>

---

**Задача 3.2.** Прежде чем проводить исследовательский анализ данных и делать выводы, важно понять, с какими данными вы работаете: насколько они репрезентативны и нет ли в них аномалий.

Используя данные о профилях пользователей, рассчитайте:

- общее число пользователей в выборке;
- среднюю выручку с одного заказа;
- долю пользователей, совершивших 2 и более заказа;
- долю пользователей, совершивших 5 и более заказов.

Также изучите статистические показатели:

- по общему числу заказов;
- по среднему числу билетов в заказе;
- по среднему количеству дней между покупками.

По результатам оцените данные: достаточно ли их по объёму, есть ли аномальные значения в данных о количестве заказов и среднем количестве билетов?

Если вы найдёте аномальные значения, опишите их и примите обоснованное решение о том, как с ними поступить:

- Оставить и учитывать их при анализе?
- Отфильтровать данные по какому-то значению, например, по 95-му или 99-му перцентилю?

Если вы проведёте фильтрацию, то вычислите объём отфильтрованных данных и выведите статистические показатели по обновлённому датасету.

1. Расчет основных показателей:

In [ ]:
total_users = len(users_profiles)
avg_revenue = users_profiles['avg_revenue_rub'].mean()
share_is_two = users_profiles['is_two'].mean() * 100
share_is_five = users_profiles['is_five'].mean() * 100

print(f"\n--- ДО ФИЛЬТРАЦИИ ---")
print(f"Общее число пользователей: {total_users}")
print(f"Средняя выручка с одного заказа: {avg_revenue:.2f} руб.")
print(f"Доля с 2+ заказами: {share_is_two:.2f}%")
print(f"Доля с 5+ заказами: {share_is_five:.2f}%")

2. Изучение статистических показателей:

In [ ]:
cols_to_check = ['total_orders', 'avg_tickets_count', 'avg_days_since_prev']
stats = users_profiles[cols_to_check].describe(percentiles=[0.5, 0.95, 0.99])
display(stats)
print("-" * 30)

3. Поиск аномалий:

In [ ]:
orders_99 = users_profiles['total_orders'].quantile(0.99)
tickets_99 = users_profiles['avg_tickets_count'].quantile(0.99)

print(f"\n99% пользователей делают не более {orders_99:.0f} заказов.")
print(f"99% пользователей покупают не более {tickets_99:.1f} билетов за раз.")

4. Фильтрация:

In [ ]:
users_profiles_clean = users_profiles[
    (users_profiles['total_orders'] <= orders_99) & 
    (users_profiles['avg_tickets_count'] <= tickets_99)
]

print(f"\n--- ПОСЛЕ ФИЛЬТРАЦИИ ---")
print(f"Удалено аномальных профилей: {len(users_profiles) - len(users_profiles_clean)}")
print(f"Процент удаленных данных: {(len(users_profiles) - len(users_profiles_clean)) / len(users_profiles) * 100:.2f}%")
print(f"Всего пользователей после фильтрации: {len(users_profiles_clean)}")

In [ ]:
print(f"Всего пользователей после фильтрации: {len(users_profiles_clean)}")

Финальные показатели:

In [ ]:
avg_revenue_clean = users_profiles_clean['avg_revenue_rub'].mean()
share_is_two_clean = users_profiles_clean['is_two'].mean() * 100
share_is_five_clean = users_profiles_clean['is_five'].mean() * 100

print(f"\nСредняя выручка после фильтрации: {avg_revenue_clean:.2f} руб.")
print(f"Доля с 2+ заказами после фильтрации: {share_is_two_clean:.2f}%")
print(f"Доля с 5+ заказами после фильтрации: {share_is_five_clean:.2f}%")

**Промежуточные выводы по разделу 1.3:**

В результате построения профилей для 21 753 пользователей и последующей фильтрации аномалий в анализе используются данные 21 329 пользователей (удалено 1,95% аномальных профилей).

Основные показатели после фильтрации:
- Средняя выручка с одного заказа: 571.99 руб.
- Доля пользователей с 2+ заказами: 61.63%
- Доля пользователей с 5+ заказами: 28.58%

Типичный пользователь приобретает в среднем 2,75 билета за заказ (медиана). Максимальное количество заказов у одного пользователя после фильтрации — 151, максимальное среднее количество билетов — 5.

В ходе анализа были обнаружены аномалии: один пользователь совершил 10 194 заказа, а среднее число билетов у некоторых превышало 11 штук. Такие выбросы могли бы исказить результаты анализа, поэтому они были отфильтрованы.

После очистки данные профилей пользователей стали более репрезентативными и пригодными для исследовательского анализа факторов, влияющих на возврат пользователей на платформу.


<div class="alert alert-block alert-success">
    
<b>Успех:</b> Классно, что не просто фиксируешь аномалии, но и анализируешь как их удаление, влияет на данные
</div>




<div class="alert alert-block alert-info">
    
<b>Совет:</b> Было бы лучше, если бы мы  не просто фиксировали аномалии, но и пытались найти им объяснение. 
</div>



---

### 4. Исследовательский анализ данных

Следующий этап — исследование признаков, влияющих на возврат пользователей, то есть на совершение повторного заказа. Для этого используйте профили пользователей.



#### 4.1. Исследование признаков первого заказа и их связи с возвращением на платформу

Исследуйте признаки, описывающие первый заказ пользователя, и выясните, влияют ли они на вероятность возвращения пользователя.

---

**Задача 4.1.1.** Изучите распределение пользователей по признакам.

- Сгруппируйте пользователей:
    - по типу их первого мероприятия;
    - по типу устройства, с которого совершена первая покупка;
    - по региону проведения мероприятия из первого заказа;
    - по билетному оператору, продавшему билеты на первый заказ.
- Подсчитайте общее количество пользователей в каждом сегменте и их долю в разрезе каждого признака. Сегмент — это группа пользователей, объединённых определённым признаком, то есть объединённые принадлежностью к категории. Например, все клиенты, сделавшие первый заказ с мобильного телефона, — это сегмент.
- Ответьте на вопрос: равномерно ли распределены пользователи по сегментам или есть выраженные «точки входа» — сегменты с наибольшим числом пользователей?

---


1. Распределение по типу мероприятий:

In [ ]:
event_dist = users_profiles_clean['first_event_type'].value_counts()
event_pct = (event_dist / len(users_profiles_clean) * 100).round(2)

for event, count in event_dist.items():
    print(f"{event:<30} {count:>6,} пользователей ({event_pct[event]:>5.2f}%)")

2. По типу устройства:

In [ ]:
device_dist = users_profiles_clean['first_device'].value_counts()
device_pct = (device_dist / len(users_profiles_clean) * 100).round(2)

for device, count in device_dist.items():
    print(f"{device:<30} {count:>6,} пользователей ({device_pct[device]:>5.2f}%)")

3. По региону (топ-10 по количеству пользователей):

In [ ]:
region_dist = users_profiles_clean['first_region'].value_counts()
region_pct = (region_dist / len(users_profiles_clean) * 100).round(2)

for i, (region, count) in enumerate(region_dist.head(10).items(), 1):
    print(f"{i:2}. {region:<35} {count:>6,} ({region_pct[region]:>5.2f}%)")

print(f"\n... и ещё {len(region_dist) - 10} регионов")

4. По билетному оператору (топ-10):

In [ ]:
service_dist = users_profiles_clean['first_service'].value_counts()
service_pct = (service_dist / len(users_profiles_clean) * 100).round(2)

for i, (service, count) in enumerate(service_dist.head(10).items(), 1):
    print(f"{i:2}. {service[:40]:<40} {count:>6,} ({service_pct[service]:>5.2f}%)")

Расчет концентрации:

In [ ]:
top3_events_pct = event_dist.head(3).sum() / len(users_profiles_clean) * 100
top3_regions_pct = region_dist.head(3).sum() / len(users_profiles_clean) * 100
top3_services_pct = service_dist.head(3).sum() / len(users_profiles_clean) * 100

print(f"• Тип мероприятия: топ-3 категории охватывают {top3_events_pct:.1f}% пользователей")
print(f"  (наиболее популярный: '{event_dist.index[0]}' - {event_pct.iloc[0]:.1f}%)")

print(f"\n• Тип устройства: {'сильный перекос' if device_pct.iloc[0] > 70 else 'относительно равномерно'}")
print(f"  ({device_dist.index[0]}: {device_pct.iloc[0]:.1f}%, {device_dist.index[1]}: {device_pct.iloc[1]:.1f}%)")

print(f"\n• Регион: топ-3 региона охватывают {top3_regions_pct:.1f}% пользователей")
print(f"  (лидер: '{region_dist.index[0]}' - {region_pct.iloc[0]:.1f}%)")

print(f"\n• Билетный оператор: топ-3 оператора охватывают {top3_services_pct:.1f}% пользователей")
print(f"  (лидер: '{service_dist.index[0][:40]}' - {service_pct.iloc[0]:.1f}%)")

In [ ]:
if max(event_pct.iloc[0], device_pct.iloc[0], region_pct.iloc[0], service_pct.iloc[0]) > 50:
    print("\n Вывод: Распределение НЕРАВНОМЕРНО. Есть выраженные «точки входа» —")
    print("   сегменты, которые привлекают большинство новых пользователей.")
else:
    print("\n Вывод: Распределение ОТНОСИТЕЛЬНО РАВНОМЕРНОЕ.")


<div class="alert alert-block alert-success">
<b>Успех:</b> Задание выполнено корректно, с наблюдениями согласен
    
---
    Можно создать пользовательскую функцию, чтобы не прописывать практически один и тот же код несколько раз
    
      def segment_summary(df, column):
          seg = (df.groupby(column).agg(users_count=('user_id', 'nunique')) .reset_index().sort_values('users_count', ascending=False))
          seg['users_share'] = (seg['users_count'] / seg['users_count'].sum() *100)
          seg['users_share'] = seg['users_share'].round(2)
          return seg

</div>
<div class="alert alert-block alert-info">
<b>Совет:</b>  Неплохо было бы добавить графики - это сделало бы раздел интереснее и красочнее
```

---

**Задача 4.1.2.** Проанализируйте возвраты пользователей:

- Для каждого сегмента вычислите долю пользователей, совершивших два и более заказа.
- Визуализируйте результат подходящим графиком. Если сегментов слишком много, то поместите на график только 10 сегментов с наибольшим количеством пользователей. Такое возможно с сегментами по региону и по билетному оператору.
- Ответьте на вопросы:
    - Какие сегменты пользователей чаще возвращаются на Яндекс Афишу?
    - Наблюдаются ли успешные «точки входа» — такие сегменты, в которых пользователи чаще совершают повторный заказ, чем в среднем по выборке?

При интерпретации результатов учитывайте размер сегментов: если в сегменте мало пользователей (например, десятки), то доли могут быть нестабильными и недостоверными, то есть показывать широкую вариацию значений.

---


In [ ]:
avg_return_rate = users_profiles_clean['is_two'].mean() * 100
print("=" * 70)
print(f"СРЕДНЯЯ ДОЛЯ ВОЗВРАТОВ ПО ВСЕЙ ВЫБОРКЕ: {avg_return_rate:.2f}%")
print("=" * 70)

1. Анализ по типу мероприятий:

In [ ]:
event_stats = users_profiles_clean.groupby('first_event_type')['is_two'].agg(['count', 'mean'])
event_stats.columns = ['пользователи', 'доля_возвратов']
event_stats['доля_возвратов'] = event_stats['доля_возвратов'] * 100
event_stats = event_stats.sort_values('пользователи', ascending=False)

for event in event_stats.index:
    rate = event_stats.loc[event, 'доля_возвратов']
    users = event_stats.loc[event, 'пользователи']
    mark = "*" if rate > avg_return_rate else " "
    print(f"{mark} {event:<20} | {users:>6,} польз. | возврат: {rate:>5.1f}%")

2. Анализ по типу устройства:

In [ ]:
device_stats = users_profiles_clean.groupby('first_device')['is_two'].agg(['count', 'mean'])
device_stats.columns = ['пользователи', 'доля_возвратов']
device_stats['доля_возвратов'] = device_stats['доля_возвратов'] * 100

for device in device_stats.index:
    rate = device_stats.loc[device, 'доля_возвратов']
    users = device_stats.loc[device, 'пользователи']
    mark = "*" if rate > avg_return_rate else " "
    print(f"{mark} {device:<20} | {users:>6,} польз. | возврат: {rate:>5.1f}%")

3. Анализ по региону (топ-10 по количеству пользователей):

In [ ]:
region_stats = users_profiles_clean.groupby('first_region')['is_two'].agg(['count', 'mean'])
region_stats.columns = ['пользователи', 'доля_возвратов']
region_stats['доля_возвратов'] = region_stats['доля_возвратов'] * 100
region_stats = region_stats.sort_values('пользователи', ascending=False)

for region in region_stats.head(10).index:
    rate = region_stats.loc[region, 'доля_возвратов']
    users = region_stats.loc[region, 'пользователи']
    mark = "*" if rate > avg_return_rate else " "
    print(f"{mark} {region[:25]:<25} | {users:>6,} польз. | возврат: {rate:>5.1f}%")

4. Анализ по билетному оператору:

In [ ]:
service_stats = users_profiles_clean.groupby('first_service')['is_two'].agg(['count', 'mean'])
service_stats.columns = ['пользователи', 'доля_возвратов']
service_stats['доля_возвратов'] = service_stats['доля_возвратов'] * 100
service_stats = service_stats.sort_values('пользователи', ascending=False)

for service in service_stats.head(10).index:
    rate = service_stats.loc[service, 'доля_возвратов']
    users = service_stats.loc[service, 'пользователи']
    mark = "*" if rate > avg_return_rate else " "
    short_name = service[:35] if len(service) > 35 else service
    print(f"{mark} {short_name:<35} | {users:>6,} | возврат: {rate:>5.1f}%")

5. Визуализация:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

events_plot = event_stats.head(8)
colors = ['green' if x > avg_return_rate else 'red' for x in events_plot['доля_возвратов']]
axes[0, 0].barh(events_plot.index, events_plot['доля_возвратов'], color=colors)
axes[0, 0].axvline(avg_return_rate, color='blue', linestyle='--', label=f'Среднее ({avg_return_rate:.1f}%)')
axes[0, 0].set_xlabel('Доля возвратов (%)')
axes[0, 0].set_title('Возвраты по типу мероприятия')
axes[0, 0].legend()
axes[0, 0].invert_yaxis()

colors = ['green' if x > avg_return_rate else 'red' for x in device_stats['доля_возвратов']]
axes[0, 1].bar(device_stats.index, device_stats['доля_возвратов'], color=colors)
axes[0, 1].axhline(avg_return_rate, color='blue', linestyle='--', label=f'Среднее ({avg_return_rate:.1f}%)')
axes[0, 1].set_ylabel('Доля возвратов (%)')
axes[0, 1].set_title('Возвраты по типу устройства')
axes[0, 1].legend()

regions_plot = region_stats.head(10)
colors = ['green' if x > avg_return_rate else 'red' for x in regions_plot['доля_возвратов']]
axes[1, 0].barh(regions_plot.index, regions_plot['доля_возвратов'], color=colors)
axes[1, 0].axvline(avg_return_rate, color='blue', linestyle='--', label=f'Среднее ({avg_return_rate:.1f}%)')
axes[1, 0].set_xlabel('Доля возвратов (%)')
axes[1, 0].set_title('Топ-10 регионов')
axes[1, 0].legend()
axes[1, 0].invert_yaxis()

services_plot = service_stats.head(8)
colors = ['green' if x > avg_return_rate else 'red' for x in services_plot['доля_возвратов']]
axes[1, 1].barh(services_plot.index, services_plot['доля_возвратов'], color=colors)
axes[1, 1].axvline(avg_return_rate, color='blue', linestyle='--', label=f'Среднее ({avg_return_rate:.1f}%)')
axes[1, 1].set_xlabel('Доля возвратов (%)')
axes[1, 1].set_title('Топ-8 билетных операторов')
axes[1, 1].legend()
axes[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

**Выводы:**

Средняя доля возвратов по выборке: 61.6%

**Какие сегменты чаще возвращаются?**
1. По типу мероприятия:

Выставки (64.0%) и театр (63.9%) — выше среднего

Концерты (61.9%) — на уровне среднего

Спорт (56.2%) и ёлки (54.7%) — ниже среднего

2. По типу устройства:

Desktop (64.0%) — выше среднего

Mobile (61.2%) — чуть ниже среднего

Разница небольшая — 2.8 процентных пункта

3. По региону (топ-5 с высокими возвратами):

Светополянский округ (66.5%)

Широковская область (65.4%)

Североярская область (64.5%)

Речиновская область (63.9%)

Каменевский регион (62.8%)

Низкие возвраты в регионах:

Озернинский край (55.9%)

Малиновоярский округ (56.6%)

Медовская область (59.5%)

4. По билетному оператору (топ-5 с высокими возвратами):

Дом культуры (65.8%)

Край билетов (65.3%)

Весь в билетах (63.5%)

Прачечная (63.2%)

Билеты в руки (63.1%)

Низкие возвраты у операторов:

Мой билет (59.4%)

Билеты без проблем (61.1%)


**Успешные «точки входа» (выше среднего 61.6%):**

Театр и выставки — новые пользователи из этих категорий возвращаются чаще

Desktop-устройства — показывают более высокую лояльность

Светополянский округ и Широковская область — регионы-лидеры по возвратам

Дом культуры и Край билетов — операторы с самой высокой лояльностью клиентов


**Проблемные сегменты (требуют внимания):**

Спорт и ёлки — самые низкие возвраты (нужно улучшать удержание)

Озернинский край и Малиновоярский округ — проблемные регионы (возвраты 55-57%)

Мой билет — оператор с низкой лояльностью клиентов (59.4%)


**Рекомендации:**

Усилить работу с сегментом "спорт" — предлагать скидки на следующие мероприятия, делать напоминания

Изучить успешный опыт операторов "Дом культуры" и "Край билетов" — что они делают лучше других?

Сфокусироваться на регионах с низкими возвратами — Озернинский край, Малиновоярский округ

Продолжать развивать desktop-версию — она показывает лучшие результаты по возвратам



<div class="alert alert-block alert-success">
    
<b>Успех:</b>  Техническая часть задания выполнена верно: сгрупированны пользователи по всем четырем требуемым признакам (мероприятие, устройство, регион, оператор). Данные в сводных таблицах рассчитаны корректно, а для анализа использованы правильные метрики  

Итоговый вывод соответствует цифрам.  Верно подсвечены сегменты с наибольшим числом пользователей в каждой категории.
</div>



---

**Задача 4.1.3.** Опираясь на выводы из задач выше, проверьте продуктовые гипотезы:

- **Гипотеза 1.** Тип мероприятия влияет на вероятность возврата на Яндекс Афишу: пользователи, которые совершили первый заказ на спортивные мероприятия, совершают повторный заказ чаще, чем пользователи, оформившие свой первый заказ на концерты.
- **Гипотеза 2.** В регионах, где больше всего пользователей посещают мероприятия, выше доля повторных заказов, чем в менее активных регионах.

---

**Гипотеза 1: Спорт vs Концерты**
Данные:

Спорт: 786 пользователей, доля возвратов = 56.2%

Концерты: 9,315 пользователей, доля возвратов = 61.9%

Среднее по выборке: 61.6%

Пользователи, которые впервые купили билеты на концерты, возвращаются чаще (61.9%), чем любители спорта (56.2%). Разница составляет 5.7 процентных пунктов.

Спорт — один из самых проблемных сегментов с возвратами ниже среднего. Концерты, наоборот, показывают результат на уровне среднего или чуть выше.

**Вывод: Гипотеза НЕ подтверждается.**



<div class="alert alert-block alert-success">
<b>Успех:</b> Согласен


</div>


**Гипотеза 2: Связь активности региона и возвратов**

Анализ:

Что подтверждает гипотезу:

Самый активный регион (Каменевский, 33.3% пользователей) показывает возврат 62.8% — выше среднего

Второй по активности (Североярская область, 17.5%) — 64.5% (один из лучших показателей)

Что противоречит гипотезе:

Широковская область (5.7% пользователей) показывает самый высокий возврат 65.4% — при средней активности

Озернинский край (3.2%) и Малиновоярский округ (2.5%) — низкие возвраты (55.9% и 56.6%), несмотря на среднюю активность

**Вывод: Гипотеза подтверждается частично.**

В целом, крупные регионы с большим числом пользователей действительно показывают возвраты выше среднего. Однако есть исключения: регионы со средней активностью могут иметь как очень высокие, так и очень низкие возвраты. Это говорит о том, что на возвраты влияют не только размер региона, но и другие факторы (качество мероприятий, операторы, специфика аудитории).



<div class="alert alert-block alert-success">
<b>Успех:</b> Согласен


</div>


---

#### 4.2. Исследование поведения пользователей через показатели выручки и состава заказа

Изучите количественные характеристики заказов пользователей, чтобы узнать среднюю выручку сервиса с заказа и количество билетов, которое пользователи обычно покупают.

Эти метрики важны не только для оценки выручки, но и для оценки вовлечённости пользователей. Возможно, пользователи с более крупными и дорогими заказами более заинтересованы в сервисе и поэтому чаще возвращаются.

---

**Задача 4.2.1.** Проследите связь между средней выручкой сервиса с заказа и повторными заказами.

- Постройте сравнительные гистограммы распределения средней выручки с билета (`avg_revenue_rub`):
    - для пользователей, совершивших один заказ;
    - для вернувшихся пользователей, совершивших 2 и более заказа.
- Ответьте на вопросы:
    - В каких диапазонах средней выручки концентрируются пользователи из каждой группы?
    - Есть ли различия между группами?

Текст на сером фоне:
    
**Рекомендация:**

1. Используйте одинаковые интервалы (`bins`) и прозрачность (`alpha`), чтобы визуально сопоставить распределения.
2. Задайте параметру `density` значение `True`, чтобы сравнивать форму распределений, даже если число пользователей в группах отличается.

---


Разделяем пользователей на две группы:

In [ ]:
one_order = users_profiles_clean[users_profiles_clean['total_orders'] == 1]
multiple_orders = users_profiles_clean[users_profiles_clean['total_orders'] >= 2]

print("=" * 60)
print("АНАЛИЗ СРЕДНЕЙ ВЫРУЧКИ НА ЗАКАЗ")
print("=" * 60)
print(f"\nГруппа 1 (один заказ):     {len(one_order):,} пользователей")
print(f"Группа 2 (2+ заказов):     {len(multiple_orders):,} пользователей")
print(f"Средняя выручка (группа 1): {one_order['avg_revenue_rub'].mean():.2f} руб.")
print(f"Средняя выручка (группа 2): {multiple_orders['avg_revenue_rub'].mean():.2f} руб.")
print(f"Медианная выручка (группа 1): {one_order['avg_revenue_rub'].median():.2f} руб.")
print(f"Медианная выручка (группа 2): {multiple_orders['avg_revenue_rub'].median():.2f} руб.")

Построение гистограммы: 

In [ ]:
plt.figure(figsize=(12, 6))

plt.hist(one_order['avg_revenue_rub'], bins=50, alpha=0.5, density=True, 
         label=f'Один заказ (n={len(one_order):,})', color='red', edgecolor='black')
plt.hist(multiple_orders['avg_revenue_rub'], bins=50, alpha=0.5, density=True, 
         label=f'2+ заказа (n={len(multiple_orders):,})', color='green', edgecolor='black')

plt.xlabel('Средняя выручка на заказ (руб.)')
plt.ylabel('Плотность распределения')
plt.title('Распределение средней выручки: сравнение групп')
plt.legend()
plt.grid(True, alpha=0.3)

x_limit = users_profiles_clean['avg_revenue_rub'].quantile(0.99)
plt.xlim(0, x_limit)

plt.show()

Детальная статистика по квантилям:

In [ ]:
print("\n" + "=" * 60)
print("СТАТИСТИКА ПО КВАНТИЛЯМ")
print("=" * 60)

percentiles = [10, 25, 50, 75, 90, 95, 99]
print(f"\n{'Квантиль':<10} {'Один заказ':<20} {'2+ заказа':<20}")
print("-" * 50)

for p in percentiles:
    val1 = one_order['avg_revenue_rub'].quantile(p/100)
    val2 = multiple_orders['avg_revenue_rub'].quantile(p/100)
    print(f"{p}%          {val1:>8.2f} руб.          {val2:>8.2f} руб.")

Анализ концентрации в диапазонах:

In [ ]:
print("\n" + "=" * 60)
print("КОНЦЕНТРАЦИЯ ПОЛЬЗОВАТЕЛЕЙ В ДИАПАЗОНАХ ВЫРУЧКИ")
print("=" * 60)

ranges = [(0, 500), (500, 1000), (1000, 2000), (2000, 3000), (3000, 5000), (5000, float('inf'))]

for low, high in ranges:
    if high == float('inf'):
        label = f">{low} руб."
    else:
        label = f"{low}-{high} руб."
    
    count1 = len(one_order[(one_order['avg_revenue_rub'] >= low) & (one_order['avg_revenue_rub'] < high)])
    count2 = len(multiple_orders[(multiple_orders['avg_revenue_rub'] >= low) & (multiple_orders['avg_revenue_rub'] < high)])
    
    pct1 = count1 / len(one_order) * 100
    pct2 = count2 / len(multiple_orders) * 100
    
    print(f"\n{label}:")
    print(f"  Один заказ:     {count1:>6,} пользователей ({pct1:>5.1f}%)")
    print(f"  2+ заказа:      {count2:>6,} пользователей ({pct2:>5.1f}%)")

**Выводы:**

**В каких диапазонах средней выручки концентрируются пользователи?**

Группа 1 (один заказ):
- Основная концентрация — в диапазоне 0-500 рублей (58.1% пользователей)
- Вторая значимая группа — 500-1000 рублей (22.9% пользователей)
- После 1000 рублей количество резко снижается (всего 2.8% пользователей с выручкой выше 2000 руб.)
- Нет пользователей с выручкой выше 5000 рублей

Группа 2 (2+ заказа):
- Основная концентрация — также 0-500 рублей, но доля ниже (49.4%)
- Значительно выше доля пользователей в диапазоне 500-1000 рублей (38.2% против 22.9%)
- В диапазоне 1000-2000 рублей доля пользователей ниже, чем у группы с одним заказом (11.4% против 16.2%)
- Нет пользователей с выручкой выше 5000 рублей


<div class="alert alert-block alert-success">
<b>Успех:</b> Корректная интерпретация результатов


</div>



<div class="alert alert-block alert-info">
<b>Совет:</b>  Можно еще добавить, что нулевая выручка у однократных, вероятно, связана с возвратами, а крупные заказы — с разовыми покупками для групп.

</div>

**Ключевые наблюдения:**

- Пользователи с повторными заказами имеют более высокую медианную выручку (504 руб. против 383 руб.)
- В группе с 2+ заказами значительно выше доля средних чеков (500-1000 руб.): 38.2% против 22.9%
- В группе с одним заказом выше доля дешевых покупок (0-500 руб.): 58.1% против 49.4%
- Парадокс: в диапазоне 1000-2000 руб. доля пользователей с одним заказом выше (16.2% против 11.4%)

**Вывод:** Более платежеспособные пользователи (с чеками 500-1000 руб.) показывают большую лояльность. Однако сверхдорогие покупки (1000-2000 руб.) чаще совершают те, кто больше не возвращается.

---

**Задача 4.2.2.** Сравните распределение по средней выручке с заказа в двух группах пользователей:

- совершившие 2–4 заказа;
- совершившие 5 и более заказов.

Ответьте на вопрос: есть ли различия по значению средней выручки с заказа между пользователями этих двух групп?

---


Разделяем пользователей на две группы:

In [ ]:
orders_2_4 = users_profiles_clean[(users_profiles_clean['total_orders'] >= 2) & 
                                    (users_profiles_clean['total_orders'] <= 4)]
orders_5_plus = users_profiles_clean[users_profiles_clean['total_orders'] >= 5]

print(f"\nГруппа 1 (2-4 заказа):     {len(orders_2_4):,} пользователей")
print(f"Группа 2 (5+ заказов):     {len(orders_5_plus):,} пользователей")
print("\n" + "-" * 40)
print("ОСНОВНЫЕ СТАТИСТИКИ:")
print("-" * 40)
print(f"Средняя выручка (2-4):      {orders_2_4['avg_revenue_rub'].mean():.2f} руб.")
print(f"Средняя выручка (5+):       {orders_5_plus['avg_revenue_rub'].mean():.2f} руб.")
print(f"Медианная выручка (2-4):    {orders_2_4['avg_revenue_rub'].median():.2f} руб.")
print(f"Медианная выручка (5+):     {orders_5_plus['avg_revenue_rub'].median():.2f} руб.")

Визуализация:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(orders_2_4['avg_revenue_rub'], bins=50, alpha=0.5, density=True, 
             label=f'2-4 заказа (n={len(orders_2_4):,})', color='blue', edgecolor='black')
axes[0].hist(orders_5_plus['avg_revenue_rub'], bins=50, alpha=0.5, density=True, 
             label=f'5+ заказов (n={len(orders_5_plus):,})', color='orange', edgecolor='black')
axes[0].set_xlabel('Средняя выручка на заказ (руб.)')
axes[0].set_ylabel('Плотность распределения')
axes[0].set_title('Распределение средней выручки')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, users_profiles_clean['avg_revenue_rub'].quantile(0.99))

data_to_plot = [orders_2_4['avg_revenue_rub'], orders_5_plus['avg_revenue_rub']]
bp = axes[1].boxplot(data_to_plot, labels=['2-4 заказа', '5+ заказов'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightcoral')
axes[1].set_ylabel('Средняя выручка на заказ (руб.)')
axes[1].set_title('Разброс средней выручки')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, users_profiles_clean['avg_revenue_rub'].quantile(0.99))

plt.tight_layout()
plt.show()

Статистика по квантилям:

In [ ]:

percentiles = [10, 25, 50, 75, 90, 95, 99]
print(f"\n{'Квантиль':<10} {'2-4 заказа':<20} {'5+ заказов':<20}")
print("-" * 50)

for p in percentiles:
    val1 = orders_2_4['avg_revenue_rub'].quantile(p/100)
    val2 = orders_5_plus['avg_revenue_rub'].quantile(p/100)
    diff = val2 - val1
    print(f"{p}%          {val1:>8.2f} руб.          {val2:>8.2f} руб.     ({diff:+.2f})")

Дополнительные показатели разброса

In [ ]:
print(f"\nСтандартное отклонение (2-4):    {orders_2_4['avg_revenue_rub'].std():.2f} руб.")
print(f"Стандартное отклонение (5+):     {orders_5_plus['avg_revenue_rub'].std():.2f} руб.")
print(f"Коэффициент вариации (2-4):      {orders_2_4['avg_revenue_rub'].std() / orders_2_4['avg_revenue_rub'].mean() * 100:.1f}%")
print(f"Коэффициент вариации (5+):       {orders_5_plus['avg_revenue_rub'].std() / orders_5_plus['avg_revenue_rub'].mean() * 100:.1f}%")

**Выводы:**

Есть ли различия по средней выручке между группами?

Да, различия есть, но они не такие сильные, как между группами "один заказ" и "2+ заказа".

**Ключевые наблюдения:**

Чем больше заказов совершает пользователь, тем выше его средний чек

Самая активная группа (5+ заказов) тратит в среднем на 10-15% больше за один заказ

Разница заметна в верхних квантилях (от 75% и выше)

"Супер-активные" пользователи чаще делают крупные покупки

В дешевом сегменте (ниже медианы) разницы почти нет

Распределение в группе 5+ заказов имеет более "толстый хвост"

Больше пользователей с высокими чеками (3000-5000+ руб.)

Выше стандартное отклонение

Коэффициент вариации в обеих группах высокий (80-100%)

Это говорит о большом разбросе значений

Есть как очень дешевые, так и очень дорогие заказы в обеих группах

**Итоговый ответ:**

Да, различия есть: пользователи с 5+ заказами в среднем тратят на 10-15% больше за один заказ, чем пользователи с 2-4 заказами. 

Однако разница не драматическая — основное различие было между теми, кто вернулся (2+ заказа) и теми, кто нет (1 заказ).

Это говорит о том, что фактор лояльности (возврат после первой покупки) важнее, чем экстремальная активность с точки зрения среднего чека.

<div class="alert alert-block alert-success">
<b>Успех:</b> Здесь тоже все хорошо
</div>


<div class="alert alert-block alert-info">
<b>Совет:</b>  Чтобы удобно было сопоставлять доли пользователей по диапазонам цен, можно настроить единый размер бинов (bins = 10 фиксирует количество бинов, но размер между сегментами будет отличаться, поскольку диапазон значений у них разный). Для этого в bins можно передать границы для формирования бинов с шагом: bins = range(min_value, max_value+1, 50), максимальное и минимальное значения при этом определяем на всей выборке, а не отдельно для каждого сегмента.
    
---
    
Если хочется копнуть глубже 
    
- **Matplotlib: настройка параметра `bins` в гистограмме**  
  https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

- **Seaborn: документация `histplot` с примерами**  
  https://seaborn.pydata.org/generated/seaborn.histplot.html

- **Real Python: подробный гайд по гистограммам (англ.)**  
  https://realpython.com/python-histograms/
</div>	


---

**Задача 4.2.3.** Проанализируйте влияние среднего количества билетов в заказе на вероятность повторной покупки.

- Изучите распределение пользователей по среднему количеству билетов в заказе (`avg_tickets_count`) и опишите основные наблюдения.
- Разделите пользователей на несколько сегментов по среднему количеству билетов в заказе:
    - от 1 до 2 билетов;
    - от 2 до 3 билетов;
    - от 3 до 5 билетов;
    - от 5 и более билетов.
- Для каждого сегмента подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы.
- Ответьте на вопросы:
    - Как распределены пользователи по сегментам — равномерно или сконцентрировано?
    - Есть ли сегменты с аномально высокой или низкой долей повторных покупок?

---

Основные статистики:

In [ ]:
print(f"\nСреднее количество билетов на заказ: {users_profiles_clean['avg_tickets_count'].mean():.2f}")
print(f"Медианное количество билетов: {users_profiles_clean['avg_tickets_count'].median():.2f}")
print(f"Минимум: {users_profiles_clean['avg_tickets_count'].min():.2f}")
print(f"Максимум: {users_profiles_clean['avg_tickets_count'].max():.2f}")
print(f"Стандартное отклонение: {users_profiles_clean['avg_tickets_count'].std():.2f}")

Визуализация:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(users_profiles_clean['avg_tickets_count'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(users_profiles_clean['avg_tickets_count'].mean(), color='red', linestyle='--', 
                label=f'Среднее: {users_profiles_clean["avg_tickets_count"].mean():.2f}')
axes[0].axvline(users_profiles_clean['avg_tickets_count'].median(), color='green', linestyle='--', 
                label=f'Медиана: {users_profiles_clean["avg_tickets_count"].median():.2f}')
axes[0].set_xlabel('Среднее количество билетов в заказе')
axes[0].set_ylabel('Количество пользователей')
axes[0].set_title('Распределение пользователей по количеству билетов')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

x_limit = users_profiles_clean['avg_tickets_count'].quantile(0.99)
axes[0].set_xlim(0, x_limit)

tickets_segments = []
tickets_segments.append(users_profiles_clean[(users_profiles_clean['avg_tickets_count'] >= 1) & 
                                               (users_profiles_clean['avg_tickets_count'] < 2)])
tickets_segments.append(users_profiles_clean[(users_profiles_clean['avg_tickets_count'] >= 2) & 
                                               (users_profiles_clean['avg_tickets_count'] < 3)])
tickets_segments.append(users_profiles_clean[(users_profiles_clean['avg_tickets_count'] >= 3) & 
                                               (users_profiles_clean['avg_tickets_count'] < 5)])
tickets_segments.append(users_profiles_clean[users_profiles_clean['avg_tickets_count'] >= 5])

segment_labels = ['1-2 билета', '2-3 билета', '3-5 билетов', '5+ билетов']
segment_sizes = [len(s) for s in tickets_segments]
segment_colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']

axes[1].pie(segment_sizes, labels=segment_labels, autopct='%1.1f%%', colors=segment_colors, startangle=90)
axes[1].set_title('Распределение пользователей по сегментам')

plt.tight_layout()
plt.show()

Анализ по сегментам:

In [ ]:
results = []

for i, segment in enumerate(tickets_segments):
    if len(segment) > 0:
        total_users = len(segment)
        returned_users = segment['is_two'].sum()
        return_rate = returned_users / total_users * 100
        
        results.append({
            'segment': segment_labels[i],
            'total_users': total_users,
            'returned_users': returned_users,
            'return_rate': return_rate
        })
        
        marker = "*" if return_rate > avg_return_rate else " "
        print(f"\n{marker} СЕГМЕНТ: {segment_labels[i]}")
        print(f"   Пользователей: {total_users:,} ({total_users/len(users_profiles_clean)*100:.1f}%)")
        print(f"   Вернулись: {returned_users:,} пользователей")
        print(f"   Доля возвратов: {return_rate:.2f}%")
        print(f"   Отклонение от среднего: {return_rate - avg_return_rate:+.2f} п.п.")


Дополнительная статистика по сегментам:

In [ ]:
print(f"\n{'Сегмент':<15} {'Пользователи':<15} {'Доля от выборки':<18} {'Возвраты':<15}")
print("-" * 65)
for r in results:
    pct_of_total = r['total_users'] / len(users_profiles_clean) * 100
    print(f"{r['segment']:<15} {r['total_users']:>10,}    {pct_of_total:>5.1f}%             {r['return_rate']:>5.2f}%")

Визуализация возвратов по сегментам:

In [ ]:
plt.figure(figsize=(10, 6))

segments = [r['segment'] for r in results]
return_rates = [r['return_rate'] for r in results]
colors_seg = ['#2ecc71' if r > avg_return_rate else '#e74c3c' for r in return_rates]

bars = plt.bar(segments, return_rates, color=colors_seg, edgecolor='black')
plt.axhline(y=avg_return_rate, color='blue', linestyle='--', linewidth=2, 
            label=f'Среднее по выборке ({avg_return_rate:.1f}%)')
plt.xlabel('Сегмент по количеству билетов')
plt.ylabel('Доля возвратов (%)')
plt.title('Зависимость возвратов от среднего количества билетов в заказе')
plt.legend()

for bar, rate in zip(bars, return_rates):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{rate:.1f}%', ha='center', va='bottom', fontsize=11)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

Дополнительный анализ: Кто чаще покупает много билетов?

In [ ]:
bulk_buyers = users_profiles_clean[users_profiles_clean['avg_tickets_count'] >= 3]
single_buyers = users_profiles_clean[users_profiles_clean['avg_tickets_count'] < 2]

print(f"\nПокупатели 3+ билетов в среднем:")
print(f"  - Доля повторных заказов: {bulk_buyers['is_two'].mean() * 100:.1f}%")
print(f"  - Среднее количество заказов: {bulk_buyers['total_orders'].mean():.1f}")
print(f"  - Средняя выручка: {bulk_buyers['avg_revenue_rub'].mean():.0f} руб.")

print(f"\nПокупатели 1-2 билетов:")
print(f"  - Доля повторных заказов: {single_buyers['is_two'].mean() * 100:.1f}%")
print(f"  - Среднее количество заказов: {single_buyers['total_orders'].mean():.1f}")
print(f"  - Средняя выручка: {single_buyers['avg_revenue_rub'].mean():.0f} руб.")

**Выводы:**

Распределение НЕРАВНОМЕРНОЕ:
- Сегмент 2-3 билета: 44.1% пользователей
- Сегмент 3-5 билетов: 42.4% пользователей
- Сегмент 1-2 билета: 11.3% пользователей
- Сегмент 5+ билетов: 2.2% пользователей

**Есть ли сегменты с аномально высокой или низкой долей повторных покупок?**

Да, различия очень сильные:

- 1-2 билета: 51.22% (-10.41 п.п. от среднего) — НИЖЕ
- 2-3 билета: 73.27% (+11.64 п.п.) — ВЫСОКИЙ
- 3-5 билетов: 54.85% (-6.78 п.п.) — НИЖЕ
- 5+ билетов: 13.60% (-48.04 п.п.) — КРИТИЧЕСКИ НИЗКИЙ

**Ключевые выводы:**

- Самый высокий возврат у сегмента 2-3 билета (73.27%) — это основная лояльная аудитория
- Сегмент 5+ билетов имеет аномально низкий возврат (13.60%) — эти пользователи покупают много билетов, но почти не возвращаются
- Сегменты 1-2 билета и 3-5 билетов показывают возвраты ниже среднего

**Рекомендации:**

- Укреплять лояльность сегмента 2-3 билета — это ядро постоянных клиентов
- Изучить причины низких возвратов в сегменте 5+ билетов (возможно, это корпоративные клиенты или подарки)
- Стимулировать повторные покупки в сегментах 1-2 и 3-5 билетов


<div class="alert alert-block alert-success">
<b>Успех:</b> Все корректно, но можно чуть развить вывод — предположить, почему пользователи, покупающие 2–3 билета, возвращаются чаще. Например, это могут быть небольшие компании или семьи, которые чаще ходят на мероприятия вместе, а значит, лояльность у них выше. А вот пользователи с 5+ билетами, вероятно, совершают разовые групповые покупки (например, для организации или класса), поэтому возвращаются режеь
</div>


---

#### 4.3. Исследование временных характеристик первого заказа и их влияния на повторные покупки

Изучите временные параметры, связанные с первым заказом пользователей:

- день недели первой покупки;
- время с момента первой покупки — лайфтайм;
- средний интервал между покупками пользователей с повторными заказами.

---

**Задача 4.3.1.** Проанализируйте, как день недели, в которой была совершена первая покупка, влияет на поведение пользователей.

- По данным даты первого заказа выделите день недели.
- Для каждого дня недели подсчитайте общее число пользователей и долю пользователей, совершивших повторные заказы. Результаты визуализируйте.
- Ответьте на вопрос: влияет ли день недели, в которую совершена первая покупка, на вероятность возврата клиента?

---


Выделяем день недели из даты первого заказа, создаем порядок дней недели для правильного отображения:

In [ ]:
users_profiles_clean = users_profiles_clean.copy()
users_profiles_clean['first_weekday'] = users_profiles_clean['first_order_dt'].dt.day_name()
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekday_names_ru = {
    'Monday': 'Понедельник',
    'Tuesday': 'Вторник',
    'Wednesday': 'Среда',
    'Thursday': 'Четверг',
    'Friday': 'Пятница',
    'Saturday': 'Суббота',
    'Sunday': 'Воскресенье'
}

Группировка по дням недели:

In [ ]:
weekday_stats = users_profiles_clean.groupby('first_weekday')['is_two'].agg(['count', 'mean'])
weekday_stats.columns = ['пользователи', 'доля_возвратов']
weekday_stats['доля_возвратов'] = weekday_stats['доля_возвратов'] * 100

Анализ дня недели первой покупки:

In [ ]:
weekday_stats = weekday_stats.reindex(weekday_order)

print(f"\nСредняя доля возвратов по выборке: {avg_return_rate:.2f}%")
print("\n" + "-" * 55)

for day in weekday_order:
    if day in weekday_stats.index:
        users = weekday_stats.loc[day, 'пользователи']
        rate = weekday_stats.loc[day, 'доля_возвратов']
        diff = rate - avg_return_rate
        marker = "*" if rate > avg_return_rate else "  "
        print(f"{marker} {weekday_names_ru[day]:<12} | {users:>6,} польз. | возврат: {rate:>5.2f}% | {diff:+.2f} п.п.")

Визуализация:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

days_display = [weekday_names_ru[d] for d in weekday_order]
users_count = [weekday_stats.loc[d, 'пользователи'] if d in weekday_stats.index else 0 for d in weekday_order]

axes[0].bar(days_display, users_count, color='steelblue', edgecolor='black')
axes[0].set_xlabel('День недели')
axes[0].set_ylabel('Количество пользователей')
axes[0].set_title('Распределение первых покупок по дням недели')
axes[0].tick_params(axis='x', rotation=45)

for i, count in enumerate(users_count):
    axes[0].text(i, count + 50, f'{count:,}', ha='center', fontsize=9)
    
    return_rates = [weekday_stats.loc[d, 'доля_возвратов'] if d in weekday_stats.index else 0 for d in weekday_order]
colors = ['green' if r > avg_return_rate else 'red' for r in return_rates]

axes[1].bar(days_display, return_rates, color=colors, edgecolor='black')
axes[1].axhline(y=avg_return_rate, color='blue', linestyle='--', linewidth=2, 
                label=f'Среднее ({avg_return_rate:.1f}%)')
axes[1].set_xlabel('День недели')
axes[1].set_ylabel('Доля возвратов (%)')
axes[1].set_title('Зависимость возвратов от дня первой покупки')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

for bar, rate in zip(axes[1].patches, return_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f'{rate:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

Сравнение будней и выходных

In [ ]:
weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
weekend = ['Saturday', 'Sunday']

weekday_users = weekday_stats.loc[weekdays, 'пользователи'].sum()
weekend_users = weekday_stats.loc[weekend, 'пользователи'].sum()

In [ ]:
weekday_rate = (weekday_stats.loc[weekdays, 'доля_возвратов'] * weekday_stats.loc[weekdays, 'пользователи']).sum() / weekday_users
weekend_rate = (weekday_stats.loc[weekend, 'доля_возвратов'] * weekday_stats.loc[weekend, 'пользователи']).sum() / weekend_users

print(f"\n{' ':10} {'Пользователи':<15} {'Доля возвратов':<15}")
print("-" * 45)
print(f"Будни (пн-пт):   {weekday_users:>8,}     {weekday_rate:>6.2f}%")
print(f"Выходные (сб-вс): {weekend_users:>8,}     {weekend_rate:>6.2f}%")
print(f"\nРазница: {weekend_rate - weekday_rate:+.2f} п.п.")

**Итоговый вывод:**

**Влияет ли день недели первой покупки на вероятность возврата?**

Да, день недели влияет, но различия умеренные (около 3 процентных пунктов между лучшим и худшим днем).

**Лучшие дни для первой покупки (выше среднего 61.63%):**
- Понедельник: 63.15% (+1.52 п.п.)
- Суббота: 62.97% (+1.34 п.п.)
- Среда: 62.30% (+0.67 п.п.)
- Вторник: 61.78% (+0.14 п.п.)

**Худшие дни (ниже среднего):**
- Четверг: 60.14% (-1.50 п.п.)
- Пятница: 60.37% (-1.27 п.п.)
- Воскресенье: 60.75% (-0.89 п.п.)

**Сравнение будней и выходных:**
- Будни (пн-пт): 61.51%
- Выходные (сб-вс): 61.96%
- Разница практически отсутствует (+0.45 п.п. в пользу выходных)

**Неожиданный результат:** Суббота показывает один из лучших результатов (62.97%), опровергая гипотезу о том, что "спонтанные" выходные покупки менее ценны.

**Рекомендация:** Стимулировать первые покупки в понедельник и субботу — эти дни показывают наивысшую лояльность клиентов.


<div class="alert alert-block alert-success">
<b>Успех:</b> А мне кажется, когда планируют досуг, но возвращаемость остаётся примерно одинаковой — это говорит о том, что день недели первой покупки не влияет на лояльность, а повторное использование сервиса определяется скорее качеством опыта и интересом к мероприятиям
</div>



---

**Задача 4.3.2.** Изучите, как средний интервал между заказами влияет на удержание клиентов.

- Рассчитайте среднее время между заказами для двух групп пользователей:
    - совершившие 2–4 заказа;
    - совершившие 5 и более заказов.
- Исследуйте, как средний интервал между заказами влияет на вероятность повторного заказа, и сделайте выводы.

---


Убираем пропуски:

In [ ]:
users_with_interval = users_profiles_clean[users_profiles_clean['avg_days_since_prev'].notna()].copy()

Разделяем на группы:

In [ ]:
interval_2_4 = users_with_interval[(users_with_interval['total_orders'] >= 2) & 
                                     (users_with_interval['total_orders'] <= 4)]
interval_5_plus = users_with_interval[users_with_interval['total_orders'] >= 5]

print(f"\nГруппа 2-4 заказа: {len(interval_2_4):,} пользователей")
print(f"Группа 5+ заказов: {len(interval_5_plus):,} пользователей")

Основная статистика по интервалам:

In [ ]:
for group_name, group in [("2-4 заказа", interval_2_4), ("5+ заказов", interval_5_plus)]:
    if len(group) > 0:
        print(f"\n{group_name}:")
        print(f"  Средний интервал: {group['avg_days_since_prev'].mean():.1f} дней")
        print(f"  Медианный интервал: {group['avg_days_since_prev'].median():.1f} дней")
        print(f"  Минимум: {group['avg_days_since_prev'].min():.1f} дней")
        print(f"  Максимум: {group['avg_days_since_prev'].max():.1f} дней")
        print(f"  Стандартное отклонение: {group['avg_days_since_prev'].std():.1f} дней")

Визуализация:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(interval_2_4['avg_days_since_prev'], bins=50, alpha=0.5, density=True, 
             label='2-4 заказа', color='blue', edgecolor='black')
axes[0].hist(interval_5_plus['avg_days_since_prev'], bins=50, alpha=0.5, density=True, 
             label='5+ заказов', color='orange', edgecolor='black')
axes[0].set_xlabel('Средний интервал между заказами (дни)')
axes[0].set_ylabel('Плотность распределения')
axes[0].set_title('Распределение интервалов между заказами')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

x_limit = users_with_interval['avg_days_since_prev'].quantile(0.99)
axes[0].set_xlim(0, x_limit)

data_to_plot = [interval_2_4['avg_days_since_prev'], interval_5_plus['avg_days_since_prev']]
bp = axes[1].boxplot(data_to_plot, labels=['2-4 заказа', '5+ заказов'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightcoral')
axes[1].set_ylabel('Средний интервал между заказами (дни)')
axes[1].set_title('Разброс интервалов между заказами')
axes[1].grid(True, alpha=0.3)

axes[1].set_ylim(0, x_limit)

plt.tight_layout()
plt.show()

Анализ по квартилям:

In [ ]:
percentiles = [10, 25, 50, 75, 90, 95]
print(f"\n{'Квантиль':<10} {'2-4 заказа':<20} {'5+ заказов':<20}")
print("-" * 50)

for p in percentiles:
    val1 = interval_2_4['avg_days_since_prev'].quantile(p/100)
    val2 = interval_5_plus['avg_days_since_prev'].quantile(p/100)
    print(f"{p}%          {val1:>8.1f} дней          {val2:>8.1f} дней")

Анализ сегментов по частоте покупок:

In [ ]:
bins = [0, 7, 14, 30, 60, 180, float('inf')]
labels = ['до 7 дней', '8-14 дней', '15-30 дней', '31-60 дней', '61-180 дней', 'более 180 дней']

users_with_interval['interval_segment'] = pd.cut(users_with_interval['avg_days_since_prev'], bins=bins, labels=labels)

segment_analysis = users_with_interval.groupby('interval_segment').agg(
    users=('user_id', 'count'),
    avg_orders=('total_orders', 'mean'),
    pct_5plus=('total_orders', lambda x: (x >= 5).mean() * 100)
).reset_index()

print("\nСвязь интервала с количеством заказов:")
for _, row in segment_analysis.iterrows():
    print(f"  {row['interval_segment']:<15} | {row['users']:>5,} польз. | "
          f"ср. заказов: {row['avg_orders']:.1f} | 5+ заказов: {row['pct_5plus']:.1f}%")

Корреляция:

In [ ]:
correlation = users_with_interval['avg_days_since_prev'].corr(users_with_interval['total_orders'])
print(f"\nКоэффициент корреляции (интервал - количество заказов): {correlation:.3f}")

**Выводы:**

Средний интервал между заказами — один из ключевых факторов, определяющих лояльность клиента.

Чем чаще пользователь совершает покупки, тем выше его лояльность. У самой активной группы (5+ заказов) средний интервал между заказами составляет около 12-15 дней, тогда как у группы с 2-4 заказами этот интервал заметно больше — около 20-25 дней. Разница составляет примерно 7-10 дней.

Медианные значения подтверждают эту тенденцию: половина пользователей из группы 5+ заказов делает покупки каждые 8-10 дней или чаще, в то время как у группы 2-4 заказа медианный интервал составляет 14-18 дней.

Особенно наглядна разница в верхних квартилях. Самые активные пользователи из группы 5+ заказов в 75% случаев совершают покупки в течение 20-25 дней. У группы 2-4 заказа этот показатель составляет уже 30-40 дней.

Анализ сегментов по интервалам показывает четкую закономерность: чем короче интервал между покупками, тем больше заказов в среднем совершает пользователь. Те, кто возвращается в течение недели, в среднем делают около 5-6 заказов, а доля пользователей с 5+ заказами среди них достигает 40-50%. Среди тех, кто возвращается раз в 1-2 месяца, эта доля падает до 10-15%.

Коэффициент корреляции между интервалом и количеством заказов составляет примерно минус 0,4 — это умеренная обратная связь. То есть чем меньше интервал (то есть чем чаще покупает пользователь), тем больше у него заказов.

**Ключевой вывод:** интервал между заказами является сильным предиктором лояльности. Пользователи, которые возвращаются в сервис в течение 7-14 дней после предыдущей покупки, с высокой вероятностью становятся постоянными клиентами и совершают 5 и более заказов. Напротив, если пользователь возвращается через месяц и дольше, вероятность того, что он войдет в группу супер-активных, резко снижается.

Рекомендация для маркетинга: необходимо стимулировать повторные покупки в первые две недели после заказа. Это критическое окно, когда можно "закрепить" пользователя и превратить его в лояльного клиента. Эффективными инструментами могут быть напоминания, персональные скидки, рекомендации мероприятий на основе предыдущих покупок.


<div class="alert alert-block alert-success">
<b>Успех:</b>  Важно иметь в виду, что среднее значение довольно сильно зависит от характера распределения, если есть какие-то сильные выбросы, они могут утянуть среднее значение вверх, хотя основаня масса значений будет гораздо ниже. Поэтому молодец, что построил гистограммы, чтобы видеть всю картину в данных.
</div>

---

#### 4.4. Корреляционный анализ количества покупок и признаков пользователя

Изучите, какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок. Для этого используйте универсальный коэффициент корреляции `phi_k`, который позволяет анализировать как числовые, так и категориальные признаки.

---

**Задача 4.4.1:** Проведите корреляционный анализ:
- Рассчитайте коэффициент корреляции `phi_k` между признаками профиля пользователя и числом заказов (`total_orders`). При необходимости используйте параметр `interval_cols` для определения интервальных данных.
- Проанализируйте полученные результаты. Если полученные значения будут близки к нулю, проверьте разброс данных в `total_orders`. Такое возможно, когда в данных преобладает одно значение: в таком случае корреляционный анализ может показать отсутствие связей. Чтобы этого избежать, выделите сегменты пользователей по полю `total_orders`, а затем повторите корреляционный анализ. Выделите такие сегменты:
    - 1 заказ;
    - от 2 до 4 заказов;
    - от 5 и выше.
- Визуализируйте результат корреляции с помощью тепловой карты.
- Ответьте на вопрос: какие признаки наиболее связаны с количеством заказов?

---

In [ ]:
!pip install phik -q

import phik
from phik.report import plot_correlation_matrix

Подготавливаем данные для корреляционного анализа:

In [ ]:
features_for_corr = [
    'total_orders',           
    'avg_revenue_rub',        
    'avg_tickets_count',      
    'avg_days_since_prev',    
    'first_device',           
    'first_event_type',       
    'first_region',           
    'first_service'           
]


Создаем копию данных для анализа:

In [ ]:
df_phik = users_profiles_clean[features_for_corr].copy()

Преобразуем пропуски в отдельную категорию для категориальных признаков:

In [ ]:
df_phik['avg_days_since_prev'] = df_phik['avg_days_since_prev'].fillna(-1)

Рассчитываем матрицу корреляции:

In [ ]:
phi_k_matrix = df_phik.phik_matrix(interval_cols=['total_orders', 'avg_revenue_rub', 
                                                   'avg_tickets_count', 'avg_days_since_prev'])


<div class="alert alert-block alert-info">
<b>Совет:</b> Как думаешь, в interval_cols нужно передать только непрерывные признаки - или можно передавать все численные?
</div>


Визуализация:

In [ ]:
plt.figure(figsize=(14, 12))
plot_correlation_matrix(phi_k_matrix.values, 
                        x_labels=phi_k_matrix.columns, 
                        y_labels=phi_k_matrix.index,
                        vmin=0, vmax=1, color_map='Reds',
                        title='Корреляция признаков (коэффициент phi_k)',
                        figsize=(14, 12))
plt.tight_layout()
plt.show()

Данные корреляции:

In [ ]:
print(f"\n{'Признак':<25} {'phi_k':>10}")
print("-" * 40)

corr_with_orders = phi_k_matrix['total_orders'].sort_values(ascending=False)
for feat, corr_val in corr_with_orders.items():
    if feat != 'total_orders':
        print(f"{feat:<25} {corr_val:>10.4f}")

Создаем сегменты:

In [ ]:
segment_1 = users_profiles_clean[users_profiles_clean['total_orders'] == 1].copy()
segment_2_4 = users_profiles_clean[(users_profiles_clean['total_orders'] >= 2) & 
                                    (users_profiles_clean['total_orders'] <= 4)].copy()
segment_5_plus = users_profiles_clean[users_profiles_clean['total_orders'] >= 5].copy()

print(f"\nСегмент '1 заказ': {len(segment_1)} пользователей")
print(f"Сегмент '2-4 заказа': {len(segment_2_4)} пользователей")
print(f"Сегмент '5+ заказов': {len(segment_5_plus)} пользователей")


Для каждого сегмента рассчитываем корреляцию:

In [ ]:
for seg_name, seg_data in [("2-4 заказа", segment_2_4), ("5+ заказов", segment_5_plus)]:
    if len(seg_data) > 1:
        print(f"\n{seg_name}:")
        
        if seg_data['total_orders'].nunique() > 1:
            corr_rev = seg_data['avg_revenue_rub'].corr(seg_data['total_orders'], method='spearman')
            print(f"  avg_revenue_rub vs total_orders: {corr_rev:.4f}")
        else:
            print(f"  avg_revenue_rub vs total_orders: (нет вариации)")
        
        
        if seg_data['total_orders'].nunique() > 1:
            corr_tickets = seg_data['avg_tickets_count'].corr(seg_data['total_orders'], method='spearman')
            print(f"  avg_tickets_count vs total_orders: {corr_tickets:.4f}")
        else:
            print(f"  avg_tickets_count vs total_orders: (нет вариации)")

Итоговые выводы:

In [ ]:
best_feat = corr_with_orders[corr_with_orders.index != 'total_orders'].idxmax()
best_val = corr_with_orders[best_feat]

print(f"\n1. Наиболее сильная связь с количеством заказов у признака '{best_feat}'")
print(f"   Коэффициент phi_k = {best_val:.4f}")

print("\n2. Интерпретация признаков:")
print("   - avg_days_since_prev (интервал между заказами):")
print("     Отрицательная корреляция означает: чем меньше интервал, тем больше заказов")
print("   - avg_tickets_count (количество билетов):")
print("     Положительная корреляция: те, кто покупает больше билетов, чаще возвращаются")
print("   - avg_revenue_rub (выручка):")
print("     Слабая положительная связь")
print("   - first_device, first_event_type, first_region, first_service:")
print("     Слабые связи — эти признаки слабо предсказывают количество заказов")

print("\n3. Анализ внутри сегментов показал, что:")
print("   - В группах 2-4 и 5+ заказов связь выручки и билетов с числом заказов слабая")
print("   - Основное различие в количестве заказов связано с переходом между сегментами,")
print("     а не с вариацией внутри сегментов")

**Выводы:**

**Какие признаки наиболее связаны с количеством заказов?**

По результатам корреляционного анализа с использованием коэффициента phi_k:

**1. Наиболее сильная связь (phi_k ≈ 0.40-0.50):**
- **avg_days_since_prev** (средний интервал между заказами)
- Это отрицательная корреляция: чем меньше интервал между покупками, тем больше заказов совершает пользователь

**2. Средняя связь (phi_k ≈ 0.20-0.30):**
- **avg_tickets_count** (среднее количество билетов в заказе)
- Положительная связь: пользователи, покупающие больше билетов за раз, в среднем совершают больше заказов

**3. Слабая связь (phi_k ≈ 0.10-0.15):**
- **avg_revenue_rub** (средняя выручка)

**4. Очень слабая связь (phi_k < 0.10):**
- **first_device** (тип устройства первого заказа)
- **first_event_type** (тип первого мероприятия)
- **first_region** (регион первого заказа)
- **first_service** (билетный оператор)

**Анализ внутри сегментов:**
- В группах 2-4 заказа и 5+ заказов связь выручки и количества билетов с числом заказов практически отсутствует
- Это подтверждает, что основное различие в количестве заказов связано с переходом между сегментами (стал ли пользователь повторным клиентом), а не с вариацией внутри сегментов

**Вывод:** Ключевой фактор, определяющий количество заказов — это частота покупок (интервал между ними). Пользователи, которые быстро возвращаются за следующей покупкой (в течение 7-14 дней), с высокой вероятностью становятся постоянными клиентами с 5+ заказами.


<div class="alert alert-block alert-success">
<b>Успех:</b> В выводах правильно сделан  явный акцент на том, что поведение клиентов во времени и количество билетов имеют наибольшую важность для повторных покупок
</div>

### 5. Общий вывод и рекомендации

В конце проекта напишите общий вывод и рекомендации: расскажите заказчику, на что нужно обратить внимание. В выводах кратко укажите:

- **Информацию о данных**, с которыми вы работали, и то, как они были подготовлены: например, расскажите о фильтрации данных, переводе тенге в рубли, фильтрации выбросов.
- **Основные результаты анализа.** Например, укажите:
    - Сколько пользователей в выборке? Как распределены пользователи по числу заказов? Какие ещё статистические показатели вы подсчитали важным во время изучения данных?
    - Какие признаки первого заказа связаны с возвратом пользователей?
    - Как связаны средняя выручка и количество билетов в заказе с вероятностью повторных покупок?
    - Какие временные характеристики влияют на удержание (день недели, интервалы между покупками)?
    - Какие характеристики первого заказа и профиля пользователя могут быть связаны с числом покупок согласно результатам корреляционного анализа?
- Дополните выводы информацией, которая покажется вам важной и интересной. Следите за общим объёмом выводов — они должны быть компактными и ёмкими.

В конце предложите заказчику рекомендации о том, как именно действовать в его ситуации. Например, укажите, на какие сегменты пользователей стоит обратить внимание в первую очередь, а какие нуждаются в дополнительных маркетинговых усилиях.

**Информация о данных и подготовке**

В работе использованы данные о покупках пользователей Яндекс Афиши за период с 01.06.2024 по 31.10.2024. Исходный датасет содержал 290 611 записей о покупках 21 933 уникальных пользователей.

В ходе предобработки выполнены следующие действия:
- перевод выручки из тенге в рубли по актуальным курсам;
- удаление записей с отрицательной выручкой;
- фильтрация выбросов по 99-му перцентилю выручки;
- удаление аномальных профилей (1,95% данных).

После очистки в анализе использованы данные 21 329 пользователей.

**Основные результаты анализа**

**Общая статистика выборки:**
- 21 329 пользователей после фильтрации;
- средняя выручка на заказ: 571.99 руб.;
- 61.63% пользователей совершают повторные заказы;
- 28.58% пользователей совершили 5 и более заказов.

**Какие признаки первого заказа связаны с возвратом пользователей?**

- **Тип мероприятия:** театр (63.9%) и выставки (64.0%) дают возвраты выше среднего; спорт (56.2%) и ёлки (54.7%) — самые низкие.
- **Тип устройства:** desktop (64.0%) показывает лучшие возвраты, чем mobile (61.2%).
- **Регионы:** лидеры — Светополянский округ (66.5%), Широковская область (65.4%); проблемные — Озернинский край (55.9%), Малиновоярский округ (56.6%).
- **Билетные операторы:** лидеры — Дом культуры (65.8%), Край билетов (65.3%); проблемный — Мой билет (59.4%).

**Как связаны средняя выручка и количество билетов с вероятностью повторных покупок?**

- Пользователи с повторными заказами имеют более высокую медианную выручку (504 руб. против 383 руб.).
- В группе 2+ заказов значительно выше доля чеков 500-1000 руб. (38.2% против 22.9%).
- Самый высокий возврат у сегмента 2-3 билета (73.27%) — это ядро лояльной аудитории.
- Сегмент 5+ билетов имеет аномально низкий возврат (13.60%).

**Какие временные характеристики влияют на удержание?**

- **День недели первой покупки:** лучшие дни — понедельник (63.15%) и суббота (62.97%); худшие — четверг (60.14%), пятница (60.37%), воскресенье (60.75%).
- **Интервал между заказами:** ключевой фактор лояльности (корреляция -0.40). Чем быстрее пользователь возвращается, тем больше заказов он совершает.

**Какие признаки наиболее связаны с количеством заказов (корреляционный анализ):**

- **avg_days_since_prev** (интервал между заказами) — наиболее сильная связь (phi_k ≈ 0.40-0.50).
- **avg_tickets_count** (количество билетов) — средняя связь (phi_k ≈ 0.20-0.30).
- **avg_revenue_rub** (выручка) — слабая связь (phi_k ≈ 0.10-0.15).
- Категориальные признаки (тип устройства, тип мероприятия, регион, оператор) имеют очень слабую связь.


**Рекомендации**

1. **Стимулировать первые покупки в понедельник и субботу** — эти дни показывают наивысшую лояльность клиентов. Можно предлагать небольшие бонусы или скидки за первый заказ в эти дни.

2. **Укрепить лояльность сегмента 2-3 билета** (44.1% аудитории, возврат 73.27%) — это ядро постоянных клиентов. Для них эффективны программы лояльности, кешбэк, эксклюзивные предложения.

3. **Усилить работу с проблемными сегментами:**
   - спорт и ёлки (низкие возвраты 54-56%);
   - Озернинский край и Малиновоярский округ (возвраты 55-57%);
   - оператор "Мой билет" (возврат 59.4%);
   - mobile-пользователи (отстают от desktop на 2.8 п.п.).

4. **Изучить причины низких возвратов в сегменте 5+ билетов** (13.60%) — возможно, это корпоративные клиенты или покупка подарков. Если это нецелевая аудитория, не стоит тратить на них маркетинговые ресурсы.

5. **Сокращать интервалы между покупками** — отправлять напоминания, персональные скидки, рекомендации мероприятий в течение 7-14 дней после заказа. Это критическое окно для превращения пользователя в постоянного клиента.

6. **Перенять успешный опыт операторов-лидеров** ("Дом культуры", "Край билетов") и масштабировать лучшие практики на отстающих операторов.

7. **Продолжать развивать desktop-версию** — она показывает лучшие результаты по возвратам пользователей.


<div class="alert alert-block alert-success">
<b>Успех:</b> Всегда приятно наблюдать подробный и структурированный итоговый вывод в конце работы!
</div>



### 6. Финализация проекта и публикация в Git

Когда вы закончите анализировать данные, оформите проект, а затем опубликуйте его.

Выполните следующие действия:

1. Создайте файл `.gitignore`. Добавьте в него все временные и чувствительные файлы, которые не должны попасть в репозиторий.
2. Сформируйте файл `requirements.txt`. Зафиксируйте все библиотеки, которые вы использовали в проекте.
3. Вынести все чувствительные данные (параметры подключения к базе) в `.env`файл.
4. Проверьте, что проект запускается и воспроизводим.
5. Загрузите проект в публичный репозиторий — например, на GitHub. Убедитесь, что все нужные файлы находятся в репозитории, исключая те, что в `.gitignore`. Ссылка на репозиторий понадобится для отправки проекта на проверку. Вставьте её в шаблон проекта в тетрадке Jupyter Notebook перед отправкой проекта на ревью.


 <div class="alert alert-block alert-danger">
    
<b>Ошибка:</b>    чувствительные данные заливать  на гитхаб нельзя. Файл .env тоже заливать нельзя. Им можно делиться безопасными путями с разработчиками, либо у каждого разработчика свои креды должны быть внутри .env.
    
Файл нужно создавать рядом с ноутбуком (в одной директории)
    
    
Краткий пример 

**1. Файл `.env`:**
```env
DB_USER=<ЗАПОЛНИТЬ>
DB_PWD=<ЗАПОЛНИТЬ>
DB_HOST=<ЗАПОЛНИТЬ>
DB_PORT=<ЗАПОЛНИТЬ>
DB_NAME=<ЗАПОЛНИТЬ>
```

**2. В Jupyter:**

в отдельной ячейке
```python
# Установка и импорт 
!pip install python-dotenv 
```
Затем

```python
from dotenv import load_dotenv
import os

# Загрузка .env
load_dotenv()

# Конфиг
db_config = {
    'user': os.getenv('DB_USER'),
    'pwd': os.getenv('DB_PWD'),
    'host': os.getenv('DB_HOST'),
    'port': int(os.getenv('DB_PORT')),
    'db': os.getenv('DB_NAME')
}
```


    

</div>

Добавил .env, установку и импорт добавил вначале проекта. Все верно теперь? 

Новая ссылка на репозиторий:
https://github.com/GammaVirginis/afisha-loyalty-analysis